### DTO 예제
- DTO(Data Transfer Object)를 설명할 때 가장 유명하고 이해하기 쉬운 예제가 바로 비밀번호 숨기기입니다.
- 초급자는 보통 이렇게 생각합니다. "DB에서 조회한 객체를 그대로 JSON으로 보내면 되는 것 아닌가?"
- 실무에서는 거의 그렇게 하지 않습니다.
- 왜냐하면 데이터베이스에는 외부에 공개하면 안 되는 정보가 포함되어 있기 때문입니다.

1. SQLite에 고객 정보 저장
2. Entity 객체 생성
3. Entity를 그대로 출력
4. DTO 생성
5. DTO로 변환
6. 비밀번호가 제거되는 것 확인

In [2]:
from dataclasses import dataclass, asdict
import sqlite3

In [3]:
# 데이터베이스 생성

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    email TEXT NOT NULL,
    password_hash TEXT NOT NULL
)
""")

conn.commit()
conn.close()

print("DB 생성 완료")

DB 생성 완료


In [4]:
# 샘플 데이터 저장

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute(
    """
    INSERT INTO customers(
        name,
        email,
        password_hash
    )
    VALUES (?, ?, ?)
    """,
    (
        "홍길동",
        "hong@test.com",
        "x7a9bc2de4f"
    )
)

conn.commit()
conn.close()

print("데이터 저장 완료")

데이터 저장 완료


In [5]:
# 데이터 확인

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
""")

rows = cursor.fetchall()

print(rows)

conn.close()

[(1, '홍길동', 'hong@test.com', 'x7a9bc2de4f')]


In [6]:
# Entity 정의

@dataclass
class CustomerEntity:
    id: int
    name: str
    email: str
    password_hash: str

In [7]:
# DB → Entity 변환

conn = sqlite3.connect("customer.db")

cursor = conn.cursor()

cursor.execute("""
SELECT *
FROM customers
LIMIT 1
""")

row = cursor.fetchone()   # 조회한 데이터를 저장 (2605331)

conn.close()

customer = CustomerEntity(     # 조회한 데이터를 Entity 변환
    id=row[0],
    name=row[1],
    email=row[2],
    password_hash=row[3]   # 비밀번호 : 숨겨야 할 내용 (260531)
)

customer

CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f')

In [ ]:
# Entity 그대로 출력

print(customer)   # 비밀번호가 그대로 노출되는 문제점 (260531)

CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f')


In [ ]:
asdict(customer)    # json형태로 출력 

{'id': 1,
 'name': '홍길동',
 'email': 'hong@test.com',
 'password_hash': 'x7a9bc2de4f'}

In [10]:
# DTO 정의

@dataclass
class CustomerResponseDTO:
    id: int
    name: str
    email: str

In [11]:
# Entity -> DTO 변환

response = CustomerResponseDTO(
    id=customer.id,
    name=customer.name,
    email=customer.email
)

response

CustomerResponseDTO(id=1, name='홍길동', email='hong@test.com')

In [12]:
asdict(response)

{'id': 1, 'name': '홍길동', 'email': 'hong@test.com'}

In [14]:
customer

CustomerEntity(id=1, name='홍길동', email='hong@test.com', password_hash='x7a9bc2de4f')

In [15]:
response_test =  CustomerResponseDTO(customer)
response_test

TypeError: CustomerResponseDTO.__init__() missing 2 required positional arguments: 'name' and 'email'

In [ ]:
from dataclasses import dataclass

@dataclass
class CustomerResponseDTO:
    id: int
    name: str
    email: str
    
    # Entity → DTO 변환이구나
    @classmethod                        # DTO 변환 메서드 추가 (260531) : 비밀번호만 제외하고 전달하도록 메서드 추가
    def from_entity(cls, customer):

        return cls(
            id=customer.id,
            name=customer.name,
            email=customer.email
        )

In [ ]:
response_test =  CustomerResponseDTO.from_entity(customer)    # 변환 메서드를 사용하여 생성
response_test

CustomerResponseDTO(id=1, name='홍길동', email='hong@test.com')

### DTO 변환 메서드에 @classmethod를 사용하는 이유는
1. 객체를 만들기 전에 호출할 수 있다.
2. cls(...) 형태로 새로운 객체를 생성할 수 있다.
3. 상속 구조에서도 올바르게 동작한다.
4. "Entity → DTO 변환 생성자"라는 의도를 명확히 표현한다.

- UserDTO.from_entity(user)
- OrderDTO.from_entity(order)
- CustomerDTO.from_model(customer)
- ProductDTO.from_entity(product)

- 이들은 모두 @classmethod를 사용한 대체 생성자 패턴의 대표적인 예입니다.